GSE176078 — Data loading

Loads the raw 10x-format matrix (barcodes, genes, sparse matrix) and metadata, downloaded from GEO as Wu_etal_2021_BRCA_scRNASeq, constructs an AnnData object, and saves it as GSE176078_raw.h5ad. Must be run within the activated scrna conda environment.

In [1]:
%reset -f

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.io
from scipy.sparse import csr_matrix
import anndata as ad
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
RAW_DIR = PROJECT_DIR / "Data" / "Raw"
base_path = RAW_DIR / "Wu_etal_2021_BRCA_scRNASeq"

print(sorted(p.name for p in base_path.iterdir()))

['.Rhistory', 'count_matrix_barcodes.tsv', 'count_matrix_genes.tsv', 'count_matrix_sparse.mtx', 'metadata.csv']


In [2]:
X = scipy.io.mmread(base_path / "count_matrix_sparse.mtx")
X = csr_matrix(X, dtype=np.float32)
adata = ad.AnnData(X)

genes = pd.read_csv(base_path / "count_matrix_genes.tsv", header=None, sep="\t")
barcodes = pd.read_csv(base_path / "count_matrix_barcodes.tsv", header=None, sep="\t")

# Matrix is genes x cells as loaded; transpose to standard cells x genes
adata.obs_names = genes[0].astype(str).values
adata.var_names = barcodes[0].astype(str).values
adata = adata.T
adata.var_names_make_unique()
adata.obs_names_make_unique()

print(adata)

AnnData object with n_obs × n_vars = 100064 × 29733


In [4]:
metadata = pd.read_csv(base_path / "metadata.csv")
metadata.index = metadata["Unnamed: 0"].astype(str)
adata.obs = metadata.loc[adata.obs_names].copy()
adata.obs["dataset"] = "GSE176078"

print(adata)
print(adata.obs.head())

AnnData object with n_obs × n_vars = 100064 × 29733
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major', 'dataset'
                                        Unnamed: 0 orig.ident  nCount_RNA  \
CID3586_AAGACCTCAGCATGAG  CID3586_AAGACCTCAGCATGAG    CID3586        4581   
CID3586_AAGGTTCGTAGTACCT  CID3586_AAGGTTCGTAGTACCT    CID3586        1726   
CID3586_ACCAGTAGTTGTGGCC  CID3586_ACCAGTAGTTGTGGCC    CID3586        1229   
CID3586_ACCCACTAGATGTCGG  CID3586_ACCCACTAGATGTCGG    CID3586        1352   
CID3586_ACTGATGGTCAACTGT  CID3586_ACTGATGGTCAACTGT    CID3586        1711   

                          nFeature_RNA  percent.mito subtype  \
CID3586_AAGACCTCAGCATGAG          1689      1.506221   HER2+   
CID3586_AAGGTTCGTAGTACCT           779      5.793743   HER2+   
CID3586_ACCAGTAGTTGTGGCC           514      1.383238   HER2+   
CID3586_ACCCACTAGATGTCGG           609      1.923077   HER2+   
CID358

In [5]:
adata.write(RAW_DIR / "GSE176078_raw.h5ad")
print("Saved successfully")

adata_check = sc.read_h5ad(RAW_DIR / "GSE176078_raw.h5ad")
assert adata_check.shape == (100064, 29733), "Shape mismatch — check before proceeding"
print("Shape verified — matches expected 100,064 x 29,733")

Saved successfully
Shape verified — matches expected 100,064 x 29,733
